In [2]:
import plotly.graph_objects as go
from dash import Dash, html, dcc
import plotly.express as px
from plotly.subplots import make_subplots

import geopandas as gpd
from shapely.geometry import Point
from geodatasets import get_url
import re

import pandas as pd
import numpy as np

In [3]:
societies = pd.read_csv('../D-PLACE-dplace-cldf-908f302/cldf/societies.csv')

In [4]:
#start gemini

# 2. Convert to a GeoDataFrame
geoData = gpd.GeoDataFrame(
    societies, 
    geometry=gpd.points_from_xy(societies['Longitude'], societies['Latitude']),
    crs="EPSG:4326"
)

In [5]:
# 3. Load the world map directly from the web
# Using the 110m resolution "admin_0_countries" map
world_url = "https://naciscdn.org/naturalearth/110m/cultural/ne_110m_admin_0_countries.zip"
world = gpd.read_file(world_url)

# 4. Filter to just what we need to keep it fast
world = world[['CONTINENT', 'geometry']]

In [6]:
# 5. Spatial Join
# 'predicate="within"' checks if the point is inside the continent polygon
result = gpd.sjoin(geoData, world, how="left", predicate="within")
# End Gemini

In [7]:
#just to see if there are any other "types"\
type_group = result.groupby('type').count()['ID']
type_group

type
languoid    4085
society     2599
Name: ID, dtype: int64

In [8]:
# I'm going to just focus on socieities
result_soc = result[result['type'] == 'society']
result_soc.shape

(2599, 23)

In [9]:
#Let's make an easier dataframe
SocDF = result_soc.drop(columns=['Glottocode', 'xd_id', 'HRAF_name_ID', 'comment', 'glottocode_comment', 'Language_Level_Glottocodes', 'ISO639P3code', 'Contribution_ID', 'type', 'HRAF_ID', 'origLat', 'origLong', 'geometry', 'index_right'])
#i removed 13 columns that I either didn't need/want or were redundant

In [10]:
SocDF['main_focal_year'] = pd.to_numeric(SocDF['main_focal_year'])

In [11]:
americamap = {
    "North America": "Americas",
    "South America": "Americas",
    "Africa": "Africa",
    "Asia": "Asia",
    "Europe": "Europe",
    "Oceania": "Oceania"
}

SocDF['Continent2'] = SocDF['CONTINENT'].map(americamap)

In [12]:
continent = SocDF.groupby('Continent2').count()['ID']

In [13]:
year_group = SocDF.groupby('main_focal_year').count()['ID']

In [14]:
worldepic = pd.read_csv('world-folk-epics-longlat.csv')

worldepic.head()

,Title,Description,Language,Tradition,Region,Year (Rough),Lat,Long
0,T'heydinn,"a Mauritanian epic ensemble, performed in Has...",Hassaniya,Mauritanian,African,1600.0,20.0,-12.0
1,Gassire's Lute,"an epic from the Soninke people, West Africa",Soninke,Soninke,African,400.0,15.0,-8.0
2,Bayajidda,a West African epic,Hausa,Hausa,African,900.0,12.5,7.5
3,Eri,a West African epic,Igbo,Igbo,African,900.0,6.0,7.0
4,Oduduwa,a West African epic,Yoruba,Yoruba,African,1000.0,7.5,4.5


In [15]:
epicdata = pd.DataFrame(worldepic)

epicdata.head()

,Title,Description,Language,Tradition,Region,Year (Rough),Lat,Long
0,T'heydinn,"a Mauritanian epic ensemble, performed in Has...",Hassaniya,Mauritanian,African,1600.0,20.0,-12.0
1,Gassire's Lute,"an epic from the Soninke people, West Africa",Soninke,Soninke,African,400.0,15.0,-8.0
2,Bayajidda,a West African epic,Hausa,Hausa,African,900.0,12.5,7.5
3,Eri,a West African epic,Igbo,Igbo,African,900.0,6.0,7.0
4,Oduduwa,a West African epic,Yoruba,Yoruba,African,1000.0,7.5,4.5


In [16]:
region_group = epicdata.groupby('Region').count()['Title']

region_group

Region
African                                11
Albanian                                1
American                                2
Balto-Slavic                            8
Celtic                                  8
East and Central Asian                 14
Germanic                                8
Greek                                   3
Italic/Romance                          6
Northeast Caucasian                     1
South Asian (Indian Subcontinent)      18
Southeast Asian                        13
Southwest Asian (Arabian Peninsula)    21
Uralic                                  2
Name: Title, dtype: int64

In [17]:
regionmap1 = {
    "African": "Africa",
    "Albanian": "Europe",
    "American": "Americas",
    "Balto–Slavic": "Europe",
    "Celtic": "Europe",
    "East and Central Asian": "Asia",
    "Germanic": "Europe",
    "Greek": "Europe",
    "Italic/Romance": "Europe",
    "Northeast Caucasian": "Asia/Europe",
    "South Asian (Indian Subcontinent)": "Asia",
    "Southeast Asian": "Asia",
    "Southwest Asian (Arabian Peninsula)": "Asia",
    "Uralic": "Asia/Europe"
}

In [18]:
regionmap = {
    "African": "Africa",
    "Albanian": "Europe",
    "American": "Americas",
    "Balto–Slavic": "Europe",
    "Celtic": "Europe",
    "East and Central Asian": "Asia",
    "Germanic": "Europe",
    "Greek": "Europe",
    "Italic/Romance": "Europe",
    "Northeast Caucasian": "Asia",
    "South Asian (Indian Subcontinent)": "Asia",
    "Southeast Asian": "Asia",
    "Southwest Asian (Arabian Peninsula)": "Asia",
    "Uralic": "Europe"
}

In [19]:
epicdata['Continent'] = epicdata['Region'].map(regionmap)

In [20]:
epicscontinent = epicdata.groupby('Continent').count()["Title"]

In [21]:
#A traditions vs. epics comparison
fig3 = make_subplots(rows = 1, cols = 2, specs=[[{'type':'domain'}, {'type':'domain'}]])

fig3.add_trace(
    go.Pie(
    labels = continent.index,
    values=continent.values,
    title="Traditions by Continent"),
    row=1, col=1
    )
#I should write something short about what a tradition is, how it's essentially a culture or a larger idea of a tribe, a community

fig3.add_trace(
    go.Pie(
    labels = epicscontinent.index,
    values=epicscontinent.values,
    title="World Folk Epics by Continent"),
    row=1, col=2
    )

fig3.show()
#I chose to group North and South America so the comparison would be better
#I also edited the mapping so that Uralic is European and NE Caucasia is Asian so there wouldn't be a Asian/European category
#I'm thinking idk a stacked bar chart? I think there are better comparisons

In [22]:
year_epics = epicdata.groupby('Year (Rough)').count()["Title"]

In [23]:

regions = sorted(SocDF['region'].unique())

colors = px.colors.qualitative.Dark24 # Or Light24, Bold, etc.

color_map = {region: colors[i % len(colors)] for i, region in enumerate(regions)}
#I'm doing this bit above in order to have each region maintain a unique color

layout = go.Layout(title="Traditions Globally by Year", legend_title="Region")

fig3 = go.Figure(layout=layout) 

# Loop through each unique CONTINENT to create separate traces
for r in SocDF['region'].unique():
    SocDF_sub = SocDF[SocDF['region'] == r]
    fig3.add_trace(go.Scattergeo(
                lat=SocDF_sub["Latitude"],
                lon=SocDF_sub["Longitude"],
                text = SocDF_sub["Name"],
                customdata= SocDF_sub["main_focal_year"],
                name=r,
                mode='markers',
                marker=dict(
                    size=10,
                    opacity=0.6,
                    color=color_map[r])
    ))

fig3.update_geos(projection_type="natural earth")

fig3.update_traces(
    hovertemplate="<b>%{text}</b> "\
    "<br>%{customdata}",
    mode='markers',
    marker=dict(size=10, opacity=.5)
)
#this one seems better for a timeline
fig3.show()

In [24]:
SocDF['main_focal_year'] = pd.to_numeric(SocDF['main_focal_year'])

years

NameError: name 'years' is not defined

In [ ]:
# Convert to numeric, turning errors (like text) into NaN, then drop them
SocDF['main_focal_year'] = pd.to_numeric(SocDF['main_focal_year'], errors='coerce')
# SocDF = SocDF.dropna(subset=['main_focal_year'])

# Now sort
years = sorted(SocDF['main_focal_year'].unique())

colors = px.colors.qualitative.Dark24 # Or Light24, Bold, etc.

color_map = {region: colors[i % len(colors)] for i, region in enumerate(regions)}
#I'm doing this bit above in order to have each region maintain a unique color

layout = go.Layout(title="Traditions Globally by Year First Published", legend_title="region")

fig = go.Figure(layout=layout) 

frame_data = []
# 1. Add Initial Traces (The starting point of the animation)
first_year = years[0]
for region in SocDF['region'].unique():
    # Show only data from the very first available year
    mask = (SocDF['region'] == region) & (SocDF['main_focal_year'] <= first_year)
    sub = SocDF[mask]
    
    fig.add_trace(go.Scattergeo(
        lat=sub["Latitude"],
        lon=sub["Longitude"],
        text=sub["Name"],
        customdata=sub["main_focal_year"],
        name=region,
        hovertemplate="<b>%{text}</b><br>%{customdata}",
        mode='markers',
        marker=dict(size=10, opacity=.5)
    ))

# 2. Create the Frames (One for each 'step' in time)
frames = []
for yr in years:
    frame_data = []
    for region in SocDF['region'].unique():
        # Cumulative: Show everything up to the current slider year
        mask = (SocDF['region'] == region) & (SocDF['main_focal_year'] <= yr)
        sub = SocDF[mask]
        
        frame_data.append(go.Scattergeo(
            lat=sub["Latitude"],
        lon=sub["Longitude"],
        text=sub["Name"],
        customdata=sub["main_focal_year"],
        name=region,
        hovertemplate="<b>%{text}</b> "\
            "<br>%{customdata}",
        mode='markers',
        marker=dict(size=10, opacity=.5)
        ))
    frames.append(go.Frame(data=frame_data, name=str(yr)))

fig.frames = frames

fig.update_layout(
    title="Traditions Globally by Year First Published",
    showlegend=True,
    geo=dict(
        projection_type='natural earth'
    ),
    updatemenus=[{
        "type": "buttons",
        "buttons": [{
            "label": "Play",
            "method": "animate",
            "args": [None, {"frame": {"duration": 500, "redraw": True}, "fromcurrent": True}]
        }]
    }],
    sliders=[{
        "active": 0,
        "steps": [{
            "label": str(yr),
            "method": "animate",
            "args": [[str(yr)], {
                "frame": {"duration": 100, "redraw": True}, 
                "fromcurrent": True,
                "transition": {"duration": 500},
                "mode": "immediate"}]
        } for yr in years]
    }]
)
#I think I'm going to get ride of the play button, and also try to fix the timeline scroll (bugs out if scrolled to the end)
fig.show()

KeyboardInterrupt: 

In [ ]:
na_df = SocDF[SocDF['main_focal_year'].isna()]
na_df

,ID,Name,Latitude,Longitude,Name_and_ID_in_source,alt_names_by_society,main_focal_year,region,CONTINENT,Continent2
346,CARNEIRO4_008,Vikings,59.0,11.10,Vikings,NaN,NaN,Northern Europe,Europe,Europe
347,CARNEIRO4_009,Fon,7.0,2.00,Dahomey,NaN,NaN,West Tropical Africa,Africa,Africa
348,CARNEIRO4_010,Ashanti,7.0,-2.00,Ashanti,NaN,NaN,West Tropical Africa,Africa,Africa
349,CARNEIRO4_011,Ganda,1.0,32.00,Baganda,NaN,NaN,East Tropical Africa,Africa,Africa
350,CARNEIRO4_012,Marquesans,-8.9,-140.08,Marquesans,NaN,NaN,South-Central Pacific,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
2594,WNAI168,Santa Clara,36.0,-106.30,Santa Clara Tewa (J168),Santa Clara Tewa,NaN,South-Central U.S.A.,North America,Americas
2595,WNAI169,Nambé Pueblo,35.9,-105.90,Nambe Tewa (J169),Nambe Tewa,NaN,South-Central U.S.A.,North America,Americas
2596,WNAI170,Taos,36.5,-105.50,Taos (J170),NaN,NaN,South-Central U.S.A.,North America,Americas
2597,WNAI171,Isleta,34.8,-106.70,Isleta (J171),NaN,NaN,South-Central U.S.A.,North America,Americas


In [ ]:
all_regions = sorted(SocDF['region'].unique())

# Group A: Everything with a valid year (for the animation)
animated_df = SocDF[SocDF['main_focal_year'].notna()]

# Group B: Everything missing a year (the static background)
na_df = SocDF[SocDF['main_focal_year'].isna()]

# # Create steps every 500 years
# min_yr = int(animated_df['main_focal_year'].min())
# max_yr = int(animated_df['main_focal_year'].max())
# years = np.arange(0, max_yr + 1, 250).tolist()


# # Ensure the very last year of your data is included so the map finishes
# if max_yr not in years:
#     years.append(max_yr)

years = [0, 500, 1000, 1200, 1400, 1600, 1700, 1800, 1850, 1900, 1990]

colors = px.colors.qualitative.Dark24 # Or Light24, Bold, etc.

color_map = {region: colors[i % len(colors)] for i, region in enumerate(all_regions)}
#I'm doing this bit above in order to have each region maintain a unique color

layout = go.Layout(title="Traditions Globally by Year First Published", legend_title="region")

fig = go.Figure(layout=layout)

frame_data = []

# 1. Add Initial Traces (The starting point of the animation)
for region in animated_df['region'].unique():
    # Show only data from the very first available year
    mask = (animated_df['region'] == region) & (animated_df['main_focal_year'] <= 0)
    sub = animated_df[mask]

    # Add these first so they sit in the background
    sub_na = na_df[na_df['region'] == region]
    
    fig.add_trace(go.Scattergeo(
        lat=sub_na["Latitude"],
        lon=sub_na["Longitude"],
        text=sub_na["Name"],
        name=f"{region} (Year Unknown)",
        customdata=sub_na["main_focal_year"],
        hovertemplate="<b>%{text}</b><br>%{customdata}",
        mode='markers',
        marker=dict(
            size=4, 
            opacity=0.5,
            color="lightgrey",
            symbol="diamond"
        ),
        legendgroup="Unknown", # Groups all NAs together in the legend
        showlegend=False
    ))
    fig.add_trace(go.Scattergeo(
        lat=sub["Latitude"],
        lon=sub["Longitude"],
        text=sub["Name"],
        customdata=sub["main_focal_year"],
        name=region,
        hovertemplate="<b>%{text}</b><br>%{customdata}",
        mode='markers',
        marker=dict(size=5, opacity=.2, symbol='circle', color=color_map[region]),
        showlegend=False
    ))

# 2. Create the Frames (One for each 'step' in time)
frames = []
for yr in years:
    frame_data = []

    for region in animated_df['region'].unique():
        # Cumulative: Show everything up to the current slider year
        mask = (animated_df['region'] == region) & (animated_df['main_focal_year'] <= yr)
        sub = animated_df[mask]
        
        frame_data.append(go.Scattergeo(
            lat=sub["Latitude"],
        lon=sub["Longitude"],
        text=sub["Name"],
        customdata=sub["main_focal_year"],
        name=region,
        hovertemplate="<b>%{text}</b> "\
            "<br>%{customdata}",
        mode='markers',
        marker=dict(size=5, opacity=.2, symbol='circle', color=color_map[region])
        ))
    frames.append(go.Frame(data=frame_data, name=str(yr)))

fig.frames = frames

fig.update_layout(
    title="Traditions Globally by Year Published",
    showlegend=True,
    geo=dict(
        projection_type='natural earth',
        # showland=True,
        # landcolor="rgb(243, 243, 243)",
    ),
    updatemenus=[{
        "type": "buttons",
        "buttons": [{
            "label": "Play",
            "method": "animate",
            "args": [None, {"frame": {"duration": 500, "redraw": True}, "fromcurrent": True}]
        }]
    }],
    sliders=[{
        "active": 0,
        "steps": [{
            "label": str(yr),
            "method": "animate",
            "args": [[str(yr)], {
                "frame": {"duration": 100, "redraw": True}, 
                "fromcurrent": True,
                "transition": {"duration": 500},
                "mode": "immediate"}]
        } for yr in years]
    }]
)
#I think I'm going to get ride of the play button, and also try to fix the timeline scroll (bugs out if scrolled to the end)
fig.show()

In [26]:
epicdata = epicdata.dropna(subset=['Region'])
#I believe there must just be a single empty cell row because I got an error but everything has a region

regions = sorted(epicdata['Region'].unique())

colors = px.colors.qualitative.Dark24 # Or Light24, Bold, etc.

color_map = {region: colors[i % len(colors)] for i, region in enumerate(regions)}
#I'm doing this bit above in order to have each region maintain a unique color


layout = go.Layout(title="World Folk Epics by Tradition", legend_title="Region")
                #    , height=300, margin={"r":0,"t":0,"l":0,"b":0})

fig3 = go.Figure(layout=layout) 

# Loop through each unique Region to create separate traces
for r in epicdata['Region'].unique():
    epicdata_sub = epicdata[epicdata['Region'] == r]
    fig3.add_trace(go.Scattergeo(
                lat=epicdata_sub["Lat"],
                lon=epicdata_sub["Long"],
                text= epicdata_sub["Title"],
                customdata= epicdata_sub["Description"],
                name=r,
                mode='markers',
                marker=dict(
                    size=10,
                    opacity=0.6,
                    color=color_map[r])
    ))

fig3.update_geos(projection_type="natural earth")

fig3.update_traces(
    hovertemplate="<b>%{text}</b> "\
    "<br>%{customdata}",
    mode='markers',
    marker=dict(size=10, opacity=.5)
)
fig3.show()

In [27]:
#Gemini did this from the loc_list.txt file I exported, I would have taken years honestly
#Also I still had to edit this, I noticed "Wales" was a dropped row originally, among others, sigh
region_map = {
    'African': [
        'African', 'Sub-Saharan', 'Nigeria', 'Kenya', 'Zulu', 'Bantu', 'Yoruba', 'Sudan',
        'Congo', 'Ethiopia', 'Somali', 'Mali', 'Ghana', 'Senegal', 'Tanzania', 'Madagascar'
    ],
    'Albanian': ['Albanian', 'Albania'],

    'American': [
        'American', 'North American', 'Native American', 'Inuit', 'Navajo', 'Cherokee',
        'Aztec', 'Maya', 'Inca', 'Kechua', 'Iroquois', 'Sioux', 'Apache', 'Brazil', 'Mexico', 
        'Canada', 'Jamaica', 'Antigua', "Zapotec", "Quiche"
    ],
    'Balto-Slavic': [
        'Russian Federation', 'Ukraine', 'Ukrainian', 'Belarus', 'Poland', 'Polish',
        'Czech', 'Slovak', 'Bulgaria', 'Bulgarian', 'Serbian', 'Croatian', 'Slovenian',
        'Latvia', 'Lithuania', 'Estonia', 'Slavic', 'Northern Ukrainians', 'Serbia', 'Moravia', 'Kashubians', 'Rusyns'
    ],
    'Celtic': [
    'Celtic', 
    'Wales', 'Welsh', 
    'Ireland', 'Irish', 'Gaelic', 
    'Scotland', 'Scottish', 
    'Brittany', 'Breton', 
    'Cornwall', 'Cornish', 
    'Isle of Man', 'Manx'],

    
    'East and Central Asian': [
        'Chinese', 'Japanese', 'Korean', 'Mongolian', 'Tibetan', 'Manchu', 'Taiwan', 'East Asian', 'Japan', 'China', 'Mongols'
    ],
    'Germanic': [
        'Germany', 'German', 'Iceland', 'Norway', 'Norwegian', 'Sweden', 'Swedish', 'Switzerland', 'Luxemborg',
        'Denmark', 'Danish', 'Dutch', 'Netherlands', 'Austria', 'English', 'Schleswig-Holstein',
        'Saxony', 'Bavaria', 'Lower Saxony', 'England', 'Scandinavia', 'Shetland Islands', 'Flanders', "Grimm",
        'Bartsch', 'Schoppner', 'Lyncker', "Kuhn", "Haas"
    ],
    'Greek': ['Greece', 'Greek', 'Hellenic', 'Cyprus'],

    'Italic/Romance': [
        'Italy', 'Italian', 'France', 'French', 'Spain', 'Spanish', 'Portugal', 'Portuguese',
        'Romania', 'Romance', 'Latin', 'Toscana', 'Sicily', 'Umbria', 'Marche', 'Lazio', "Perrault", "Basile"
    ],
    'Kartvelian / South Caucasian': [
        'Georgia', 'Georgian', 'Mingrelian', 'Svan', 'Laz', 'Kartvelian'
    ],
    'Levantine & North African': [
        'Levant', 'Syria', 'Lebanon', 'Palestine', 'Jordan', 'Egypt', 'Morocco', 'Tunisia',
        'Algeria', 'Libya', 'Maghreb', 'Berber'
    ],
    'Northeast Caucasian': [
        'Chechen', 'Ingush', 'Dagestan', 'Avar', 'Lezgin', 'Circassian', 'Armenians'
    ],
    'Oceanian / Pacific Islander': [
        'Oceania', 'Polynesia', 'Melanesia', 'Micronesia', 'Hawaii', 'Maori', 'Fiji',
        'Samoa', 'Tonga', 'Tuamotu', 'Gilbert Islands', 'Solomons'
    ],
    'South Asian': [
        'Indian', 'Vedic', 'Brahman', 'Hindu', 'Pakistan', 'Bengali', 'Tamil', 'Punjabi',
        'Sri Lanka', 'Nepal', 'Bangladesh', 'Panchtantra', 'Jatakas', 'Indian Literary'
    ],
    'Southeast Asian': [
        'Vietnam', 'Thailand', 'Thai', 'Indonesia', 'Malay', 'Philippines', 'Burma',
        'Myanmar', 'Cambodia', 'Laos', 'Singapore'
    ],
    'Southwest Asian': [
        'Arabian', 'Saudi', 'Yemen', 'Oman', 'Qatar', 'Kuwait', 'Emirates', 'Bedouin', 'Bedouins'
    ],
    'Turkic': [
        'Turkish', 'Turkey', 'Azerbaijan', 'Kazakh', 'Kirghiz', 'Uzbek', 'Turkmen',
        'Tatar', 'Bashkir', 'Uygur', 'Yakut', 'Tuva'
    ],
    'Uralic': [
        'Finnish', 'Finland', 'Hungarian', 'Hungary', 'Estonian', 'Sami', 'Uralic', 'Nenets'
    ],
    'West Asian / Indo-Iranian': [
        'Persian', 'Iran', 'Tajik', 'Kurdish', 'Baluchi', 'Pashto', 'Afghan', 'Persia', 'Armenia', 'Kurds',
        "Hartland", "Temme"
    ]
}

In [28]:
def process_folklore_data(filepath):
    # Load the data
    df = pd.read_csv(filepath)
    
    # Clean the column name (fixing the space issue: 'Location / Tradition')
    target_col = 'Location / Tradition'
    
    # 2. Assignment Logic
    def assign_region(loc_string):
        if pd.isna(loc_string):
            return 'Unknown'
        for region, keywords in region_map.items():
            if any(key.lower() in str(loc_string).lower() for key in keywords):
                #i love the above line! if it fits one word, it puts the region in, saves tons!
                return region
        return 'Other / Unclassified'

    # Create the new column
    df['Region_Group'] = df[target_col].apply(assign_region)
    
    # 3. Convert to GeoDataFrame (using your existing Lat/Long)
    # Removing rows without coordinates for cleaner mapping
    df = df.dropna(subset=['Latitude', 'Longitude'])
    geometry = [Point(xy) for xy in zip(df['Longitude'], df['Latitude'])]
    gdf = gpd.GeoDataFrame(df, crs="EPSG:4326", geometry=geometry)
    
    return gdf

# Apply it to your magic file
eternity = process_folklore_data('eternal-c.csv')
warning  = process_folklore_data('warnings-c.csv')
infidelity = process_folklore_data('infidelity-c.csv')
magic = process_folklore_data('magic-c.csv')
shapeshift = process_folklore_data('changeling-c.csv')
eternity.tail()
#great let's get these on a map

,Motif Reference,Motif Title,Location / Tradition,Latitude,Longitude,source,berezkin,other_data,Region_Group,geometry
3078,Ṛbhus - Book: 3 Hymn: 60,NaN,"Indian Literary Tradition (Vedic, Brahman, Pur...",25.56227,83.08594,https://www.folkloredatabase.com/db_index.php?...,NaN,NaN,American,POINT (83.08594 25.56227)
3079,Ṛbhus - Book: 4 Hymn: 33,NaN,"Indian Literary Tradition (Vedic, Brahman, Pur...",25.56227,83.08594,https://www.folkloredatabase.com/db_index.php?...,NaN,NaN,American,POINT (83.08594 25.56227)
3080,Ṛbhus - Book: 4 Hymn: 34,NaN,"Indian Literary Tradition (Vedic, Brahman, Pur...",25.56227,83.08594,https://www.folkloredatabase.com/db_index.php?...,NaN,NaN,American,POINT (83.08594 25.56227)
3081,Ṛbhus - Book: 4 Hymn: 37,NaN,"Indian Literary Tradition (Vedic, Brahman, Pur...",25.56227,83.08594,https://www.folkloredatabase.com/db_index.php?...,NaN,NaN,American,POINT (83.08594 25.56227)
3082,Ṛtu - Book: 1 Hymn: 15,NaN,"Indian Literary Tradition (Vedic, Brahman, Pur...",25.56227,83.08594,https://www.folkloredatabase.com/db_index.php?...,NaN,NaN,American,POINT (83.08594 25.56227)


In [29]:
def spatial_classification(row):
    # Only fix the ones that are currently unclassified
    if row['Region_Group'] != 'Other/Unclassified':
        return row['Region_Group']
    
    lat, lon = row['Latitude'], row['Longitude']
    
    # Simple Polygon Logic (Approximate Bounding Boxes)
    if 35 <= lat <= 72 and -25 <= lon <= 45:
        return 'Europe (Spatial)'
    if -35 <= lat <= 35 and -20 <= lon <= 50:
        return 'Africa (Spatial)'
    if 15 <= lat <= 75 and -170 <= lon <= -50:
        return 'Americas (Spatial)'
    if -10 <= lat <= 80 and 50 <= lon <= 180:
        return 'Asia & Oceania (Spatial)'
        
    return 'Other/Still Undefined'

# Apply this to your 'magic' dataframe
magic['Region_Group'] = magic.apply(spatial_classification, axis=1)
#WHY IS NOTHING HAPPENING ;_; ugh

# with pd.option_context('display.max_rows', None):
#     print(magic['Region_Group'])

In [30]:
fig8 = make_subplots(rows = 2, cols = 2, 
                    column_widths=[0.5, 0.5], # Adjusted to match 3 columns
    row_heights=[0.5, 0.5],
    specs=[[{'type':'scattergeo'}, {'type':'scattergeo'}],
           [{'type':'scattergeo'}, {'type':'scattergeo'}]],
           subplot_titles=("Eternal Tasks", "Moral Heeding/Warning", "Infidelity in Relationships", "Magic/Soorcery"))

regions = sorted(magic['Region_Group'].unique())

colors = px.colors.qualitative.Dark24 # Or Light24, Bold, etc.

color_map = {region: colors[i % len(colors)] for i, region in enumerate(regions)}
#I'm doing this bit above in order to have each region maintain a unique color

fig8.update_layout(title="Folklore Motifs by Category", legend_title="Region", showlegend=False)

# Loop through each unique Region to create separate traces
for r in eternity['Region_Group'].unique():
    eternity_sub = eternity[eternity['Region_Group'] == r]
    #eternity
    fig8.add_trace(go.Scattergeo(
                lat=eternity_sub["Latitude"],
                lon=eternity_sub["Longitude"],
                name=r,
                mode='markers',
                marker=dict(
                    size=7,
                    opacity=0.3,
                    color=color_map[r])
    ),row=1, col=1)
    
    #warnings
for r in warning['Region_Group'].unique():
    warning_sub = warning[warning['Region_Group'] == r]
    #eternity
    fig8.add_trace(go.Scattergeo(
                lat=warning_sub["Latitude"],
                lon=warning_sub["Longitude"],
                name=r,
                mode='markers',
                marker=dict(
                    size=7,
                    opacity=0.3,
                    color=color_map[r])
    ),row=1, col=2)

    #infidelity
for r in infidelity['Region_Group'].unique():
    infidelity_sub = infidelity[infidelity['Region_Group'] == r]
    fig8.add_trace(go.Scattergeo(
                lat=infidelity_sub["Latitude"],
                lon=infidelity_sub["Longitude"],
                name=r,
                mode='markers',
                marker=dict(
                    size=7,
                    opacity=0.3,
                    color=color_map[r])
    ),row=2, col=1)

    #magic
for r in magic['Region_Group'].unique():
    magic_sub = magic[magic['Region_Group'] == r]
    fig8.add_trace(go.Scattergeo(
                lat=magic_sub["Latitude"],
                lon=magic_sub["Longitude"],
                name=r,
                mode='markers',
                marker=dict(
                    size=7,
                    opacity=0.3,
                    color=color_map[r])
    ),row=2, col=2)
    
    #i removed shapeshifting, doesn't aid the overall narrative

fig8.update_geos(projection_type="natural earth")

# 1. Calculate counts for each group
counts = {
    "eternity": len(eternity),
    "warning": len(warning),
    "infidelity": len(infidelity),
    "magic": len(magic)
}

# 2. Add annotations (x and y are from 0 to 1 across the whole figure)
# Adjust the 'y' values slightly if they overlap with your map borders
fig8.add_annotation(dict(x=0.22, y=0.6, text=f"{counts['eternity']} data points", 
                         showarrow=False, xref="paper", yref="paper", font=dict(size=12)))
fig8.add_annotation(dict(x=0.78, y=0.6, text=f"{counts['warning']} data points", 
                         showarrow=False, xref="paper", yref="paper", font=dict(size=12)))
fig8.add_annotation(dict(x=0.22, y=-0.06, text=f"{counts['infidelity']} data points", 
                         showarrow=False, xref="paper", yref="paper", font=dict(size=12)))
fig8.add_annotation(dict(x=0.78, y=-0.06, text=f"{counts['magic']} data points", 
                         showarrow=False, xref="paper", yref="paper", font=dict(size=12)))

# 3. Increase bottom margin to make room for the bottom labels
fig8.update_layout(margin=dict(b=50))

fig8.show()

In [31]:
years = sorted(epicdata['Year (Rough)'].unique())
# I was suffering w the timeline aspect so I gave my code to Gemini
fig = go.Figure()

frame_data = []
# 1. Add Initial Traces (The starting point of the animation)
first_year = years[0]
for region in epicdata['Region'].unique():
    # Show only data from the very first available year
    mask = (epicdata['Region'] == region) & (epicdata['Year (Rough)'] <= first_year)
    sub = epicdata[mask]
    
    fig.add_trace(go.Scattergeo(
        lat=sub["Lat"],
        lon=sub["Long"],
        text=sub["Title"],
        customdata=sub["Description"],
        name=region,
        hovertemplate="<b>%{text}</b><br>%{customdata}",
        mode='markers',
        marker=dict(size=10, opacity=.5)
    ))

# 2. Create the Frames (One for each 'step' in time)
frames = []
for yr in years:
    frame_data = []
    for region in epicdata['Region'].unique():
        # Cumulative: Show everything up to the current slider year
        mask = (epicdata['Region'] == region) & (epicdata['Year (Rough)'] <= yr)
        sub = epicdata[mask]
        
        frame_data.append(go.Scattergeo(
            lat=sub["Lat"],
        lon=sub["Long"],
        text=sub["Title"],
        customdata=sub["Description"],
        name=region,
        hovertemplate="<b>%{text}</b> "\
            "<br>%{customdata}",
        mode='markers',
        marker=dict(size=10, opacity=.5)
        ))
    frames.append(go.Frame(data=frame_data, name=str(yr)))

fig.frames = frames

fig.update_layout(
    title="The Spread of Global Epics Over Time",
    showlegend=True,
    geo=dict(
        projection_type='natural earth',
        # showland=True,
        # landcolor="rgb(243, 243, 243)",
    ),
    updatemenus=[{
        "type": "buttons",
        "buttons": [{
            "label": "Play",
            "method": "animate",
            "args": [None, {"frame": {"duration": 500, "redraw": True}, "fromcurrent": True}]
        }]
    }],
    sliders=[{
        "active": 0,
        "steps": [{
            "label": str(yr),
            "method": "animate",
            "args": [[str(yr)], {
                "frame": {"duration": 100, "redraw": True}, 
                "fromcurrent": True,
                "transition": {"duration": 500},
                "mode": "immediate"}]
        } for yr in years]
    }]
)
#I think I'm going to get ride of the play button, and also try to fix the timeline scroll (bugs out if scrolled to the end)
fig.show()

In [32]:
app.layout = html.Div([
    
])

NameError: name 'app' is not defined